# Extraction de métadonnées de fichiers SVS avec PySpark (Google Colab)

Ce notebook installe les dépendances nécessaires, monte Google Drive, puis extrait les métadonnées de fichiers SVS (Aperio Whole Slide Images) pour les stocker dans un DataFrame PySpark.


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("truthisneverlinear/bach-breast-cancer-histology-images")

print("Path to dataset files:", path)

100%|██████████| 12.5G/12.5G [02:17<00:00, 97.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/truthisneverlinear/bach-breast-cancer-histology-images/versions/1


In [18]:
# !apt-get install tree -y

!tree -d /root/.cache/kagglehub/datasets/truthisneverlinear/bach-breast-cancer-histology-images/versions/1/

/root/.cache/kagglehub/datasets/truthisneverlinear/bach-breast-cancer-histology-images/versions/1/
├── ICIAR2018_BACH_Challenge
│   └── ICIAR2018_BACH_Challenge
│       ├── Photos
│       │   ├── Benign
│       │   ├── InSitu
│       │   ├── Invasive
│       │   └── Normal
│       └── WSI
│           ├── gt_thumbnails
│           └── thumbnails
└── ICIAR2018_BACH_Challenge_TestDataset
    └── ICIAR2018_BACH_Challenge_TestDataset
        ├── Photos
        └── WSI
            └── thumbnails

15 directories


In [23]:
!ls -la /root/.cache/kagglehub/datasets/truthisneverlinear/bach-breast-cancer-histology-images/versions/1/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/WSI/

total 6603308
drwxr-xr-x 4 root root      4096 Apr 23 18:01 .
drwxr-xr-x 4 root root      4096 Apr 23 18:00 ..
-rw-r--r-- 1 root root 285760046 Apr 23 18:00 01.svs
-rw-r--r-- 1 root root 294823026 Apr 23 18:00 02.svs
-rw-r--r-- 1 root root 198388484 Apr 23 18:00 03.svs
-rw-r--r-- 1 root root 206136124 Apr 23 18:00 04.svs
-rw-r--r-- 1 root root 223174934 Apr 23 18:00 05.svs
-rw-r--r-- 1 root root 167005732 Apr 23 18:00 06.svs
-rw-r--r-- 1 root root 229103386 Apr 23 18:00 07.svs
-rw-r--r-- 1 root root 214039006 Apr 23 18:00 08.svs
-rw-r--r-- 1 root root 215698006 Apr 23 18:00 09.svs
-rw-r--r-- 1 root root 244537850 Apr 23 18:00 10.svs
-rw-r--r-- 1 root root 299517906 Apr 23 18:00 11.svs
-rw-r--r-- 1 root root 229988718 Apr 23 18:00 12.svs
-rw-r--r-- 1 root root 222141546 Apr 23 18:00 13.svs
-rw-r--r-- 1 root root 168122724 Apr 23 18:00 14.svs
-rw-r--r-- 1 root root 210497084 Apr 23 18:00 15.svs
-rw-r--r-- 1 root root 241903442 Apr 23 18:00 16.svs
-rw-r--r-- 1 root root 248629418 Apr 23 1

In [22]:
# ── 3. Initialisation de la session Spark ────────────────────────────────────
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SVS Metadata Extractor")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"✓ Spark {spark.version} prêt.")


✓ Spark 4.0.2 prêt.


In [24]:
# ── 4. Recherche des fichiers SVS ─────────────────────────────────────────────
import os, glob

# ⚠️  Modifiez ce chemin pour pointer vers votre répertoire de fichiers SVS
SVS_ROOT = "/root/.cache/kagglehub/datasets/truthisneverlinear/bach-breast-cancer-histology-images/versions/1/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/WSI/"   # <── à adapter

svs_paths = glob.glob(os.path.join(SVS_ROOT, "**", "*.svs"), recursive=True)

print(f"✓ {len(svs_paths)} fichier(s) SVS trouvé(s) dans : {SVS_ROOT}")
for p in svs_paths[:5]:
    print("  -", p)


✓ 30 fichier(s) SVS trouvé(s) dans : /root/.cache/kagglehub/datasets/truthisneverlinear/bach-breast-cancer-histology-images/versions/1/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/WSI/
  - /root/.cache/kagglehub/datasets/truthisneverlinear/bach-breast-cancer-histology-images/versions/1/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/WSI/17.svs
  - /root/.cache/kagglehub/datasets/truthisneverlinear/bach-breast-cancer-histology-images/versions/1/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/WSI/20.svs
  - /root/.cache/kagglehub/datasets/truthisneverlinear/bach-breast-cancer-histology-images/versions/1/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/WSI/01.svs
  - /root/.cache/kagglehub/datasets/truthisneverlinear/bach-breast-cancer-histology-images/versions/1/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/WSI/A06.svs
  - /root/.cache/kagglehub/datasets/truthisneverlinear/bach-breast-cancer-histology-images/versions/1/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/WSI/1

In [31]:
!pip install openslide-python openslide-bin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 57.5 MB/s eta 0:00:0000:0100:01


In [32]:
# ── 5. Fonction d'extraction des métadonnées ──────────────────────────────────
import json
import openslide

def extract_svs_metadata(filepath: str) -> dict:
    """
    Ouvre un fichier SVS via OpenSlide et retourne un dictionnaire
    contenant ses principales métadonnées.
    """
    try:
        slide = openslide.OpenSlide(filepath)
        props = dict(slide.properties)
        width, height = slide.dimensions

        meta = {
            # ── Informations fichier ──────────────────────────────────────
            "filepath":         filepath,
            "filename":         os.path.basename(filepath),
            "filesize_mb":      round(os.path.getsize(filepath) / 1_048_576, 2),
            # ── Structure de la pyramide d'images ─────────────────────────
            "level_count":      int(slide.level_count),
            "width_px":         int(width),
            "height_px":        int(height),
            # ── Résolution physique (microns par pixel) ───────────────────
            "mpp_x":            float(props.get("openslide.mpp-x") or 0),
            "mpp_y":            float(props.get("openslide.mpp-y") or 0),
            # ── Métadonnées Aperio (spécifiques au format SVS) ────────────
            "objective_power":  props.get("openslide.objective-power"),
            "scan_date":        props.get("aperio.Date"),
            "scan_time":        props.get("aperio.Time"),
            "scanner_id":       props.get("aperio.ScanScope ID"),
            "app_mag":          props.get("aperio.AppMag"),
            "image_id":         props.get("aperio.ImageID"),
            "vendor":           props.get("openslide.vendor"),
            "background_color": props.get("openslide.background-color"),
            # ── Toutes les propriétés brutes (JSON) ───────────────────────
            "raw_properties":   json.dumps(props),
            "error":            None,
        }
        slide.close()

    except Exception as exc:
        meta = {
            "filepath": filepath,
            "filename": os.path.basename(filepath),
            "error":    str(exc),
            **{k: None for k in [
                "filesize_mb", "level_count", "width_px", "height_px",
                "mpp_x", "mpp_y", "objective_power", "scan_date", "scan_time",
                "scanner_id", "app_mag", "image_id", "vendor",
                "background_color", "raw_properties",
            ]},
        }

    return meta

print("✓ Fonction extract_svs_metadata définie.")


✓ Fonction extract_svs_metadata définie.


In [33]:
# ── 6. Création du DataFrame PySpark ─────────────────────────────────────────
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType, IntegerType,
)

schema = StructType([
    StructField("filepath",         StringType(),  True),
    StructField("filename",         StringType(),  True),
    StructField("filesize_mb",      FloatType(),   True),
    StructField("level_count",      IntegerType(), True),
    StructField("width_px",         IntegerType(), True),
    StructField("height_px",        IntegerType(), True),
    StructField("mpp_x",            FloatType(),   True),
    StructField("mpp_y",            FloatType(),   True),
    StructField("objective_power",  StringType(),  True),
    StructField("scan_date",        StringType(),  True),
    StructField("scan_time",        StringType(),  True),
    StructField("scanner_id",       StringType(),  True),
    StructField("app_mag",          StringType(),  True),
    StructField("image_id",         StringType(),  True),
    StructField("vendor",           StringType(),  True),
    StructField("background_color", StringType(),  True),
    StructField("raw_properties",   StringType(),  True),
    StructField("error",            StringType(),  True),
])

# Extraction sur le driver (OpenSlide n'est pas disponible sur les workers distants)
records = [extract_svs_metadata(p) for p in svs_paths]

rows = [
    Row(**{f.name: rec.get(f.name) for f in schema.fields})
    for rec in records
]

df_svs = spark.createDataFrame(rows, schema=schema)

print(f"✓ DataFrame créé : {df_svs.count()} ligne(s), {len(df_svs.columns)} colonne(s)")
df_svs.printSchema()


✓ DataFrame créé : 30 ligne(s), 18 colonne(s)
root
 |-- filepath: string (nullable = true)
 |-- filename: string (nullable = true)
 |-- filesize_mb: float (nullable = true)
 |-- level_count: integer (nullable = true)
 |-- width_px: integer (nullable = true)
 |-- height_px: integer (nullable = true)
 |-- mpp_x: float (nullable = true)
 |-- mpp_y: float (nullable = true)
 |-- objective_power: string (nullable = true)
 |-- scan_date: string (nullable = true)
 |-- scan_time: string (nullable = true)
 |-- scanner_id: string (nullable = true)
 |-- app_mag: string (nullable = true)
 |-- image_id: string (nullable = true)
 |-- vendor: string (nullable = true)
 |-- background_color: string (nullable = true)
 |-- raw_properties: string (nullable = true)
 |-- error: string (nullable = true)



In [36]:
# ── 7. Aperçu des métadonnées principales ─────────────────────────────────────
cols_display = [
    "filename", "filesize_mb", "level_count",
    "width_px", "height_px", "mpp_x", "mpp_y",
    "objective_power", "scan_date", "vendor", "error"
]

df_svs.select(cols_display).show(truncate=False)


+--------+-----------+-----------+--------+---------+-----+-----+---------------+------------------------+------+-----+
|filename|filesize_mb|level_count|width_px|height_px|mpp_x|mpp_y|objective_power|scan_date               |vendor|error|
+--------+-----------+-----------+--------+---------+-----+-----+---------------+------------------------+------+-----+
|17.svs  |237.11     |3          |60513   |43588    |0.5  |0.5  |20             |2014-01-23T16:20:49.177Z|aperio|NULL |
|20.svs  |170.45     |3          |56289   |36128    |0.5  |0.5  |20             |2014-01-30T10:07:59.637Z|aperio|NULL |
|01.svs  |272.52     |3          |62625   |44737    |0.5  |0.5  |20             |2013-12-05T12:49:03.69Z |aperio|NULL |
|A06.svs |220.71     |3          |62625   |36833    |0.5  |0.5  |20             |2014-01-10T16:09:51.317Z|aperio|NULL |
|18.svs  |250.93     |3          |59969   |43912    |0.5  |0.5  |20             |2014-01-23T16:41:36.987Z|aperio|NULL |
|A08.svs |224.92     |3          |58401 

In [37]:
# ── 8. Statistiques descriptives ──────────────────────────────────────────────
df_svs.select(
    "filesize_mb", "level_count", "width_px", "height_px", "mpp_x", "mpp_y"
).describe().show()


+-------+------------------+-------------------+-----------------+-----------------+-----+-----+
|summary|       filesize_mb|        level_count|         width_px|        height_px|mpp_x|mpp_y|
+-------+------------------+-------------------+-----------------+-----------------+-----+-----+
|  count|                30|                 30|               30|               30|   30|   30|
|   mean|214.79800160725912|  3.033333333333333|57458.96666666667|          41357.9|  0.5|  0.5|
| stddev|37.758675562733266|0.18257418583505544| 6166.27442700983|4183.080191223556|  0.0|  0.0|
|    min|            144.85|                  3|            44737|            30942|  0.5|  0.5|
|    max|             286.1|                  4|            70497|            47162|  0.5|  0.5|
+-------+------------------+-------------------+-----------------+-----------------+-----+-----+



In [38]:
# ── 9. Export vers Pandas pour exploration interactive ────────────────────────
df_pandas = df_svs.drop("raw_properties").toPandas()
df_pandas


,filepath,filename,filesize_mb,level_count,width_px,height_px,mpp_x,mpp_y,objective_power,scan_date,scan_time,scanner_id,app_mag,image_id,vendor,background_color,error
0,/root/.cache/kagglehub/datasets/truthisneverli...,17.svs,237.110001,3,60513,43588,0.5,0.5,20,2014-01-23T16:20:49.177Z,None,None,"20,000000",None,aperio,None,None
1,/root/.cache/kagglehub/datasets/truthisneverli...,20.svs,170.449997,3,56289,36128,0.5,0.5,20,2014-01-30T10:07:59.637Z,None,None,"20,000000",None,aperio,None,None
2,/root/.cache/kagglehub/datasets/truthisneverli...,01.svs,272.519989,3,62625,44737,0.5,0.5,20,2013-12-05T12:49:03.69Z,None,None,"20,000000",None,aperio,None,None
3,/root/.cache/kagglehub/datasets/truthisneverli...,A06.svs,220.710007,3,62625,36833,0.5,0.5,20,2014-01-10T16:09:51.317Z,None,None,"20,000000",None,aperio,None,None
4,/root/.cache/kagglehub/datasets/truthisneverli...,18.svs,250.929993,3,59969,43912,0.5,0.5,20,2014-01-23T16:41:36.987Z,None,None,"20,000000",None,aperio,None,None
5,/root/.cache/kagglehub/datasets/truthisneverli...,A08.svs,224.919998,3,58401,41057,0.5,0.5,20,2014-01-17T16:14:59.663Z,None,None,"20,000000",None,aperio,None,None
6,/root/.cache/kagglehub/datasets/truthisneverli...,07.svs,218.490005,3,58945,40970,0.5,0.5,20,2013-11-18T13:57:16.823Z,None,None,"20,000000",None,aperio,None,None
7,/root/.cache/kagglehub/datasets/truthisneverli...,14.svs,160.330002,3,44737,38697,0.5,0.5,20,2014-01-16T09:19:56.4Z,None,None,"20,000000",None,aperio,None,None
8,/root/.cache/kagglehub/datasets/truthisneverli...,A10.svs,204.869995,3,58401,39959,0.5,0.5,20,2014-01-30T09:49:50.57Z,None,None,"20,000000",None,aperio,None,None
9,/root/.cache/kagglehub/datasets/truthisneverli...,15.svs,200.750000,3,56833,39874,0.5,0.5,20,2014-01-17T16:58:31.037Z,None,None,"20,000000",None,aperio,None,None


In [45]:
# ── 10. Lecture des métadonnées d'une image SVS ──────────────────────────────

import pandas as pd

slide = openslide.OpenSlide(svs_paths[0])
df_props = pd.DataFrame(
    list(slide.properties.items()),
    columns=["clé", "valeur"]
).sort_values("clé")
slide.close()

df_props

,clé,valeur
0,aperio.AppMag,"20,000000"
1,aperio.Date,2014-01-23T16:20:49.177Z
2,aperio.Filename,ImageCollection_0000012140
3,aperio.MPP,"0,500000"
4,aperio.OriginalHeight,60513
5,aperio.OriginalWidth,43681
6,openslide.associated.thumbnail.height,737
7,openslide.associated.thumbnail.width,1024
8,openslide.comment,Aperio Image Library v12.4.0 \r\n60513x43681 [...
9,openslide.level-count,3
